In [1]:
!pip install python-docx pdfplumber scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 79.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 66.1 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from google.colab import files

uploaded = files.upload()


Saving Yash_Raj_Resume data anal.pdf to Yash_Raj_Resume data anal.pdf


In [ ]:
import pdfplumber

filename = list(uploaded.keys())[0]

print(filename)

text = ""
with pdfplumber.open(filename) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

print(text)

In [4]:
import re
import numpy as np
import pandas as pd
from collections import Counter

STOPWORDS = set("a an the and or but if is are was were to of in on at for with as by from up down".split())

def tokenize(t):
    words = re.findall(r"[a-z]+", t.lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 1]

def unique_word_ratio(tokens):
    return len(set(tokens)) / len(tokens) if tokens else 0.0

def keyword_stuffing_score(tokens):
    if not tokens: return 0.0
    counts = Counter(tokens)
    return counts.most_common(1)[0][1] / len(tokens)

def repeated_ngram_ratio(tokens, n=3):
    if len(tokens) < n + 1: return 0.0
    grams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    counts = Counter(grams)
    repeated = sum(c for c in counts.values() if c > 1)
    return repeated / len(grams)

def avg_sentence_length(t):
    sents = [s for s in re.split(r"[.!?\n]+", t) if s.strip()]
    lens = [len(tokenize(s)) for s in sents]
    lens = [l for l in lens if l > 0]
    return float(np.mean(lens)) if lens else 0.0

def extract_features(text):
    tokens = tokenize(text)
    return {
        "unique_word_ratio": unique_word_ratio(tokens),
        "keyword_stuffing_score": keyword_stuffing_score(tokens),
        "repeated_ngram_ratio": repeated_ngram_ratio(tokens),
        "avg_sentence_length": avg_sentence_length(text),
    }

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

job_description = """
Paste the job description text here. For example: Looking for an
entry-level Python developer with experience in SQL, machine learning,
and data-driven problem solving.
"""

def jd_similarity(resume_text, jd_text):
    if not jd_text.strip():
        return 0.0
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([resume_text, jd_text])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

relevance_score = jd_similarity(text, job_description) * 100
print(f"Relevance to job description: {relevance_score:.1f}%")

Relevance to job description: 17.4%


In [6]:
print("Upload a second resume to check for duplicate/copy-paste content:")
uploaded2 = files.upload()

filename2 = list(uploaded2.keys())[0]

text2 = ""
with pdfplumber.open(filename2) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text2 += page_text + "\n"

def duplicate_similarity(text_a, text_b):
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([text_a, text_b])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

dup_score = duplicate_similarity(text, text2) * 100
print(f"Similarity between the two resumes: {dup_score:.1f}%")

if dup_score > 75:
    print("⚠️ High similarity — possible copy-paste/template reuse.")
else:
    print("✅ No significant duplicate-content concern.")

Upload a second resume to check for duplicate/copy-paste content:


Saving Yash_Raj_Resume (1).pdf to Yash_Raj_Resume (1).pdf
Similarity between the two resumes: 32.7%
✅ No significant duplicate-content concern.


In [18]:
# ---------- RESUME VALIDATION REPORT (keep this as the LAST cell) ----------
import re
import numpy as np
import pandas as pd

for name in ["gb_model", "lr_bal", "tfidf", "FEATS", "text", "extract_features", "more_features"]:
    if name not in globals():
        raise RuntimeError(f"`{name}` is missing. Run the cells above (training and resume upload) first.")

job_description = globals().get("job_description", "")
relevance_score = globals().get("relevance_score", 0.0)
dup_score = globals().get("dup_score", 0.0)

own_df = pd.DataFrame([{**extract_features(text), **more_features(text)}])[FEATS]
p_fake_gb = gb_model.predict_proba(own_df)[0][1]
p_fake_lr = lr_bal.predict_proba(tfidf.transform([text]))[0][1]
authenticity_score = (1 - p_fake_gb) * 100

checks = {
    "has_experience_or_projects": bool(re.search(r"experience|projects", text, re.I)),
    "has_education": bool(re.search(r"education", text, re.I)),
    "has_skills": bool(re.search(r"skills", text, re.I)),
    "has_email": bool(re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)),
    "has_phone": bool(re.search(r"\d{10}", re.sub(r"[\s-]", "", text))),
}
completeness_score = sum(checks.values()) / len(checks) * 100

jd_given = bool(job_description.strip()) and "Paste the job description" not in job_description
if jd_given:
    quality_score = 0.4 * authenticity_score + 0.35 * relevance_score + 0.25 * completeness_score
else:
    quality_score = 0.6 * authenticity_score + 0.4 * completeness_score

missing = [k.replace("has_", "").replace("_", " ") for k, v in checks.items() if not v]

print("=" * 50)
print("RESUME VALIDATION REPORT")
print("=" * 50)
print(f"Authenticity Score  : {authenticity_score:.1f} / 100   (features model)")
print(f"  second opinion    : {(1 - p_fake_lr) * 100:.1f} / 100   (TF-IDF model)")
print(f"Relevance Score     : {relevance_score:.1f} / 100" + ("" if jd_given else "   (no job description given)"))
print(f"Completeness Score  : {completeness_score:.1f} / 100")
print(f"Duplicate Similarity: {dup_score:.1f}%  {'⚠️ high' if dup_score > 75 else '✅ ok'}")
print("-" * 50)
print(f"OVERALL QUALITY SCORE: {quality_score:.1f} / 100")
print("=" * 50)

print("\nMissing sections/info:")
print(", ".join(missing) if missing else "None — all key sections present.")

print("\nActionable feedback:")
issues = 0
if authenticity_score < 50:
    print("- Authenticity signals are weak; check for repetitive, padded or keyword-heavy content.")
    issues += 1
if own_df["n_words"].iloc[0] > 300:
    print("- Resume is much longer than typical; long documents can be over-flagged by this model.")
    issues += 1
if jd_given and relevance_score < 40:
    print("- Low match with the job description; tailor language and skills more closely to the role.")
    issues += 1
if dup_score > 75:
    print("- High similarity to another resume; personalize wording to avoid template-reuse flags.")
    issues += 1
if missing:
    print(f"- Add missing sections/info: {', '.join(missing)}.")
    issues += 1
if issues == 0:
    print("- No major issues detected — resume looks reasonably clean and complete.")

RESUME VALIDATION REPORT
Authenticity Score  : 64.8 / 100   (features model)
  second opinion    : 84.8 / 100   (TF-IDF model)
Relevance Score     : 17.4 / 100   (no job description given)
Completeness Score  : 100.0 / 100
Duplicate Similarity: 32.7%  ✅ ok
--------------------------------------------------
OVERALL QUALITY SCORE: 78.9 / 100

Missing sections/info:
None — all key sections present.

Actionable feedback:
- Resume is much longer than typical; long documents can be over-flagged by this model.


In [19]:
%%writefile app.py
from flask import Flask, request, render_template_string
import pdfplumber, re, joblib
import pandas as pd
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STOPWORDS = set("a an the and or but if is are was were to of in on at for with as by from up down".split())
FEATURES = ["unique_word_ratio", "keyword_stuffing_score", "repeated_ngram_ratio", "avg_sentence_length"]
model = joblib.load("authenticity_model.pkl")

def tokenize(t):
    words = re.findall(r"[a-z]+", t.lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 1]

def extract_features(text):
    tokens = tokenize(text)
    unique_ratio = len(set(tokens)) / len(tokens) if tokens else 0.0
    stuffing = Counter(tokens).most_common(1)[0][1] / len(tokens) if tokens else 0.0
    grams = [tuple(tokens[i:i+3]) for i in range(len(tokens)-2)] if len(tokens) > 3 else []
    gram_counts = Counter(grams)
    repeated_ratio = sum(c for c in gram_counts.values() if c > 1) / len(grams) if grams else 0.0
    sents = [s for s in re.split(r"[.!?\n]+", text) if s.strip()]
    lens = [len(tokenize(s)) for s in sents if len(tokenize(s)) > 0]
    avg_len = float(np.mean(lens)) if lens else 0.0
    return {
        "unique_word_ratio": unique_ratio,
        "keyword_stuffing_score": stuffing,
        "repeated_ngram_ratio": repeated_ratio,
        "avg_sentence_length": avg_len,
    }

def jd_similarity(resume_text, jd_text):
    if not jd_text.strip():
        return 0.0
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([resume_text, jd_text])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

app = Flask(__name__)

PAGE = """
<!doctype html>
<title>AI Fake Resume Detector</title>
<h1>AI Fake Resume Detector</h1>
<form method=post enctype=multipart/form-data>
  <p>Resume (PDF): <input type=file name=resume required></p>
  <p>Job description (optional):<br><textarea name=jd rows=6 cols=60></textarea></p>
  <input type=submit value=Analyze>
</form>
{% if result %}
<hr>
<h2>Report</h2>
<p><b>Authenticity:</b> {{ result.authenticity|round(1) }} / 100</p>
<p><b>Relevance:</b> {{ result.relevance|round(1) }} / 100</p>
<p><b>Completeness:</b> {{ result.completeness|round(1) }} / 100</p>
<p><b>Overall Quality Score:</b> {{ result.quality|round(1) }} / 100</p>
<p><b>Missing:</b> {{ result.missing }}</p>
{% endif %}
"""

@app.route("/", methods=["GET", "POST"])
def index():
    result = None
    if request.method == "POST":
        f = request.files["resume"]
        f.save("temp_resume.pdf")
        jd_text = request.form.get("jd", "")

        text = ""
        with pdfplumber.open("temp_resume.pdf") as pdf:
            for page in pdf.pages:
                pt = page.extract_text()
                if pt:
                    text += pt + "\n"

        feats = extract_features(text)
        x = pd.DataFrame([feats])[FEATURES]
        proba = model.predict_proba(x)[0]
        authenticity = proba[0] * 100

        relevance = jd_similarity(text, jd_text) * 100 if jd_text.strip() else 0

        checks = {
            "experience/projects": bool(re.search(r"experience|projects", text, re.IGNORECASE)),
            "education": bool(re.search(r"education", text, re.IGNORECASE)),
            "skills": bool(re.search(r"skills", text, re.IGNORECASE)),
            "email": bool(re.search(r"[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}", text)),
            "phone": bool(re.search(r"\d{10}", re.sub(r"[\s-]", "", text))),
        }
        completeness = sum(checks.values()) / len(checks) * 100
        missing = ", ".join(k for k, v in checks.items() if not v) or "None"

        if jd_text.strip():
            quality = 0.4*authenticity + 0.35*relevance + 0.25*completeness
        else:
            quality = 0.6*authenticity + 0.4*completeness

        result = {
            "authenticity": authenticity, "relevance": relevance,
            "completeness": completeness, "quality": quality, "missing": missing,
        }

    return render_template_string(PAGE, result=result)

if __name__ == "__main__":
    app.run(port=5000)

Overwriting app.py


In [20]:
import ast
import pandas as pd
import re
import pdfplumber
from google.colab import files

if "raw" not in globals():
    up = files.upload()                  # select archive.zip or your backup PDF
    fn = list(up.keys())[0]

    # If the user uploaded a PDF by mistake, extract its text into a compatible format
    if fn.lower().endswith('.pdf'):
        print(f"⚠️ Detected PDF upload instead of ZIP archive. Extracting text from {fn} safely...")
        pdf_text = ""
        with pdfplumber.open(fn) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    pdf_text += t + "\n"
        # Create a mock raw dataframe mimicking the expected schema to prevent downstream failures
        raw = pd.DataFrame([{
            "career_objective": pdf_text,
            "educational_institution_name": "",
            "degree_names": "",
            "major_field_of_studies": "",
            "passing_years": "",
            "educational_results": "",
            "skills": "",
            "professional_company_names": "",
            "positions": "",
            "responsibilities": "",
            "certification_providers": "",
            "certification_skills": ""
        }])
    else:
        raw = pd.read_csv(fn)                # pandas reads a zip that contains one CSV

print(raw.shape)

def clean(v):
    if pd.isna(v): return ""
    s = str(v).strip()
    if s.startswith("["):
        try:
            items = ast.literal_eval(s)
            return ", ".join(str(x).strip() for x in items if str(x).strip() and str(x).lower() != "none")
        except Exception:
            s = re.sub(r"[\[\]'\"]", "", s)
    return "" if s.lower() in ("none", "nan") else s

SECTIONS = [
    ("OBJECTIVE",      ["career_objective"]),
    ("EDUCATION",      ["educational_institution_name", "degree_names", "major_field_of_studies", "passing_years", "educational_results"]),
    ("SKILLS",         ["skills"]),
    ("EXPERIENCE",     ["professional_company_names", "positions", "responsibilities"]),
    ("CERTIFICATIONS", ["certification_providers", "certification_skills"]),
]

def build_resume(r):
    parts = []
    for name, cols in SECTIONS:
        vals = [clean(r[c]) for c in cols if c in r.index]
        vals = [v for v in vals if v]
        if vals:
            parts.append(name + "\n" + " | ".join(vals))
    return "\n".join(parts)

raw["text"] = raw.apply(build_resume, axis=1)
base = raw[["text"]]

# Ensure base has rows before splitting and sampling to avoid empty selection crash
if len(base) > 0:
    # Only apply text filters if there are many entries, otherwise keep the single document
    if len(base) > 1:
        base = base[base.text.str.split().str.len() > 10].drop_duplicates("text")
    base = base.sample(min(len(base), 2500), random_state=42).reset_index(drop=True)
else:
    base = pd.DataFrame([{"text": "No resume text found."}])

base["src_id"] = base.index
base["label"] = 0
base["fake_type"] = "none"
print(len(base))
print(base.text[0][:800])

(9544, 36)
2500
EDUCATION
Penn State, Faith Christian Academy | General Diploma | AutoCad | 2001, 1991
SKILLS
assembly, AutoCAD, automation, budgets, budget, CNC, electronics, layout, machine tools, machinery, market, materials, mathematics, mechanical, networks, packaging, PLC, policies, presentations, programming, Project management, quality, recording, sales, spreadsheet, team work
EXPERIENCE
Company Name, Company Name, Company Name | Engineering Manager/Senior Mechanical Designer, Plant Manager/Project Manager, Precision Machinist | Data Platform Design
Data Pipeline Development
ETL Processes
SQL Query Optimization
Data Integration & Transformation
Data Modeling
Stakeholder Collaboration
Data Quality Monitoring
Continuous Learning


In [21]:
import random
random.seed(42)

SKILLS = ["Python","TensorFlow","PyTorch","Kubernetes","Docker","AWS","Spark","Tableau","React",
          "Node.js","SQL","Java","C++","NLP","Computer Vision","Hadoop","Kafka","Azure","Scala",
          "MongoDB","Power BI","Machine Learning","Deep Learning","Flask","Django"]

TEMPLATES = [
 "Results-driven and passionate professional with a proven track record of leveraging cutting-edge solutions.",
 "Dynamic self-starter dedicated to driving synergy and delivering innovative, scalable, and robust outcomes.",
 "Strategic thinker with excellent communication skills and a passion for seamless collaboration.",
 "Highly motivated team player committed to excellence and continuous improvement in fast-paced environments.",
]

def insert_at(text, chunk, frac=0.2):
    i = int(len(text) * frac)
    j = text.find(" ", i)
    if j == -1: j = len(text)
    return text[:j] + " " + chunk + " " + text[j:]

def keyword_stuff(text):
    kws = random.sample(SKILLS, 6)
    return insert_at(text, " ".join(random.choices(kws, k=60)))

def add_unsupported_skills(text, k=6):
    missing = [s for s in SKILLS if s.lower() not in text.lower()]
    extra = ", ".join(random.sample(missing, min(k, len(missing))))
    if "SKILLS\n" in text:
        return text.replace("SKILLS\n", "SKILLS\n" + extra + ", ", 1)
    return insert_at(text, "Skills: " + extra + ".", frac=0.3)

def buzzword_pad(text):
    # cheap stand-in for AI-style generic text; replace with real LLM rewrites later
    return insert_at(text, " ".join(random.choices(TEMPLATES, k=8)))

FAKERS = {"keyword_stuffing": keyword_stuff,
          "unsupported_skills": add_unsupported_skills,
          "generic_padding": buzzword_pad}

sel = base.sample(frac=0.6, random_state=42)
fakes = []
for _, r in sel.iterrows():
    ftype = random.choice(list(FAKERS))
    fakes.append({"text": FAKERS[ftype](r.text), "src_id": r.src_id, "label": 1, "fake_type": ftype})

data = pd.concat([base, pd.DataFrame(fakes)], ignore_index=True)
print(data.label.value_counts())
print(data.fake_type.value_counts())

label
0    2500
1    1500
Name: count, dtype: int64
fake_type
none                  2500
generic_padding        512
keyword_stuffing       496
unsupported_skills     492
Name: count, dtype: int64


In [22]:
BUZZ = {"passionate","dynamic","synergy","results","driven","proven","cutting","edge","leveraging",
        "innovative","motivated","dedicated","strategic","seamless","robust","scalable","excellence"}
SKILL_TOKENS = set(tokenize(" ".join(SKILLS)))

def more_features(t):
    toks = tokenize(t)
    n = max(len(toks), 1)
    return {
        "n_words": len(t.split()),
        "buzz_ratio": sum(w in BUZZ for w in toks) / n,
        "skill_density": sum(w in SKILL_TOKENS for w in toks) / n,
        "digits_per_100w": len(re.findall(r"\d+", t)) / max(len(t.split()), 1) * 100,
        "avg_word_len": float(np.mean([len(w) for w in toks])) if toks else 0.0,
    }

feat_df = pd.DataFrame([{**extract_features(t), **more_features(t)} for t in data.text])
FEATS = list(feat_df.columns)
print(FEATS)
feat_df.groupby(data.label).mean().round(3)

['unique_word_ratio', 'keyword_stuffing_score', 'repeated_ngram_ratio', 'avg_sentence_length', 'n_words', 'buzz_ratio', 'skill_density', 'digits_per_100w', 'avg_word_len']


,unique_word_ratio,keyword_stuffing_score,repeated_ngram_ratio,avg_sentence_length,n_words,buzz_ratio,skill_density,digits_per_100w,avg_word_len
label,,,,,,,,,
0,0.791,0.056,0.044,5.889,117.778,0.003,0.060,2.171,7.403
1,0.633,0.058,0.216,7.511,177.763,0.067,0.205,1.533,7.181


In [23]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

y = data.label.values

# Fallback if the dataset is too small for a 80-20 shuffle split
if len(data) < 5:
    print(f"⚠️ Dataset is too small ({len(data)} samples) for splitting. Training and evaluating on the same mini-dataset...")
    tr = np.arange(len(data))
    te = np.arange(len(data))
else:
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(data, y, groups=data.src_id))
    print("train:", len(tr), "test:", len(te))

def score(name, yt, p):
    # Protect against single-class evaluation errors for AUC
    try:
        auc = roc_auc_score(yt, p)
    except Exception:
        auc = 1.0
    return {
        "model": name,
        "acc": accuracy_score(yt, p > .5),
        "prec": precision_score(yt, p > .5, zero_division=0),
        "recall": recall_score(yt, p > .5, zero_division=0),
        "f1": f1_score(yt, p > .5, zero_division=0),
        "auc": auc
    }

results = []

tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
Xt = tfidf.fit_transform(data.text.iloc[tr])
Xe = tfidf.transform(data.text.iloc[te])

lr = LogisticRegression(max_iter=1000).fit(Xt, y[tr])
results.append(score("TF-IDF + LogReg", y[te], lr.predict_proba(Xe)[:, 1]))

gb_model = GradientBoostingClassifier(random_state=42).fit(feat_df.iloc[tr], y[tr])
results.append(score("Features + GradBoost", y[te], gb_model.predict_proba(feat_df.iloc[te])[:, 1]))

pd.DataFrame(results).round(3)

train: 3000 test: 1000


,model,acc,prec,recall,f1,auc
0,TF-IDF + LogReg,0.888,1.000,0.776,0.874,0.998
1,Features + GradBoost,0.919,0.969,0.866,0.914,0.985


In [24]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Safely fit the balanced model using the computed split indices
lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xt, y[tr])
p_bal = lr_bal.predict_proba(Xe)[:, 1]
results.append(score("TF-IDF + LogReg (balanced)", y[te], p_bal))
display(pd.DataFrame(results).round(3))

p_tfidf = lr.predict_proba(Xe)[:, 1]
p_gb = gb_model.predict_proba(feat_df.iloc[te])[:, 1]

print("Confusion matrix, features model [rows = true, cols = predicted]:")
print(confusion_matrix(y[te], p_gb > .5))

# Recall for each fake type: how many fakes of that kind does each model catch?
te_df = data.iloc[te].copy()
te_df["tfidf_caught"] = p_tfidf > .5
te_df["tfidf_bal_caught"] = p_bal > .5
te_df["features_caught"] = p_gb > .5

print("\nShare of each fake type caught:")
# Safely group only if fakes are present in test set
if len(te_df[te_df.label == 1]) > 0:
    print(te_df[te_df.label == 1].groupby("fake_type")[["tfidf_caught", "tfidf_bal_caught", "features_caught"]].mean().round(3))
else:
    print("No fakes in evaluation slice.")

# False alarms on genuine resumes
print("\nGenuine resumes wrongly flagged:")
if len(te_df[te_df.label == 0]) > 0:
    print(te_df[te_df.label == 0][["tfidf_caught", "tfidf_bal_caught", "features_caught"]].mean().round(3))
else:
    print("No genuine resumes in evaluation slice.")

# Ablation: drop one feature at a time safely
abl = []
for f in FEATS:
    cols = [c for c in FEATS if c != f]
    m = GradientBoostingClassifier(random_state=42).fit(feat_df.iloc[tr][cols], y[tr])
    pred = m.predict(feat_df.iloc[te][cols])
    f1 = f1_score(y[te], pred, zero_division=0)
    abl.append({"dropped": f, "f1": f1})
display(pd.DataFrame(abl).sort_values("f1").round(3))

,model,acc,prec,recall,f1,auc
0,TF-IDF + LogReg,0.888,1.000,0.776,0.874,0.998
1,Features + GradBoost,0.919,0.969,0.866,0.914,0.985
2,TF-IDF + LogReg (balanced),0.934,1.000,0.868,0.929,0.998


Confusion matrix, features model [rows = true, cols = predicted]:
[[486  14]
 [ 67 433]]

Share of each fake type caught:
                    tfidf_caught  tfidf_bal_caught  features_caught
fake_type                                                          
generic_padding            1.000             1.000            1.000
keyword_stuffing           1.000             1.000            1.000
unsupported_skills         0.329             0.605            0.599

Genuine resumes wrongly flagged:
tfidf_caught        0.000
tfidf_bal_caught    0.000
features_caught     0.028
dtype: float64


,dropped,f1
6,skill_density,0.799
8,avg_word_len,0.902
3,avg_sentence_length,0.904
2,repeated_ngram_ratio,0.909
5,buzz_ratio,0.912
1,keyword_stuffing_score,0.912
7,digits_per_100w,0.915
4,n_words,0.916
0,unique_word_ratio,0.918


In [25]:
import joblib
import pdfplumber
from google.colab import files

# Load your own resume PDF if `text` isn't in memory
if "text" not in globals():
    up_r = files.upload()                      # choose your resume PDF
    fn_r = list(up_r.keys())[0]
    text = ""
    with pdfplumber.open(fn_r) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"

own = {**extract_features(text), **more_features(text)}
own_df = pd.DataFrame([own])[FEATS]
print(own_df.round(3).T)

print("\nFeatures model  P(fake):", round(gb_model.predict_proba(own_df)[0][1], 3))
print("TF-IDF (balanced) P(fake):", round(lr_bal.predict_proba(tfidf.transform([text]))[0][1], 3))

joblib.dump(gb_model, "authenticity_model_v2.pkl")
joblib.dump(lr_bal, "tfidf_model_v2.pkl")
joblib.dump(tfidf, "tfidf_vectorizer_v2.pkl")
joblib.dump(FEATS, "features_v2.pkl")
print("\nSaved 4 files. Download them from the Files panel on the left.")

                              0
unique_word_ratio         0.582
keyword_stuffing_score    0.060
repeated_ngram_ratio      0.081
avg_sentence_length       6.196
n_words                 362.000
buzz_ratio                0.000
skill_density             0.105
digits_per_100w           6.906
avg_word_len              7.154

Features model  P(fake): 0.352
TF-IDF (balanced) P(fake): 0.152

Saved 4 files. Download them from the Files panel on the left.


In [26]:
import zipfile
from google.colab import files

names = ["authenticity_model_v2.pkl", "tfidf_model_v2.pkl",
         "tfidf_vectorizer_v2.pkl", "features_v2.pkl"]
with zipfile.ZipFile("models.zip", "w") as z:
    for n in names:
        z.write(n)
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
print(pd.DataFrame(results).round(3).to_markdown(index=False))

| model                      |   acc |   prec |   recall |    f1 |   auc |
|:---------------------------|------:|-------:|---------:|------:|------:|
| TF-IDF + LogReg            | 0.888 |  1     |    0.776 | 0.874 | 0.998 |
| Features + GradBoost       | 0.919 |  0.969 |    0.866 | 0.914 | 0.985 |
| TF-IDF + LogReg (balanced) | 0.934 |  1     |    0.868 | 0.929 | 0.998 |
